In [1]:
import os
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import faiss
from sentence_transformers import SentenceTransformer
import warnings
import torch

# Фиксация seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ARTIFACTS_DIR = os.path.join("artifacts")
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

/home/mrmin50000/github/aie-environment/homeworks/HW14/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
docs = [
    {"id": "doc_01", "title": "Что такое Docker", "text": "Docker — это платформа для разработки, доставки и запуска приложений в контейнерах. Контейнеры изолируют приложение и его зависимости от хостовой ОС, обеспечивая воспроизводимость и переносимость."},
    {"id": "doc_02", "title": "Dockerfile", "text": "Dockerfile — это текстовый файл с инструкциями для сборки образа. Основные команды: FROM, RUN, COPY, CMD, EXPOSE. Образ создаётся послойно, каждый RUN/COPY добавляет новый слой к предыдущему."},
    {"id": "doc_03", "title": "docker build", "text": "Команда docker build собирает образ из Dockerfile. Пример: docker build -t myapp:latest . Флаг -t задаёт тег образа, точка указывает на контекст сборки с Dockerfile."},
    {"id": "doc_04", "title": "docker run", "text": "Команда docker run создаёт и запускает контейнер из образа. Параметры: -d (фоновый режим), -p 8080:80 (проброс портов), --name (имя контейнера), -v (монтирование томов)."},
    {"id": "doc_05", "title": "Docker Compose", "text": "Docker Compose позволяет описывать мультиконтейнерные приложения в YAML-файле docker-compose.yml. Команды: docker compose up (запуск), docker compose down (остановка и удаление), docker compose ps (список)."},
    {"id": "doc_06", "title": "Тома и хранилище", "text": "Данные в контейнере по умолчанию эфемерны. Для сохранения используют тома (volumes) или bind mounts. Тома управляются Docker и находятся в /var/lib/docker/volumes. Bind mounts монтируют путь с хоста."},
    {"id": "doc_07", "title": "Сети в Docker", "text": "Контейнеры изолированы в сетях по умолчанию. Команды: docker network create, docker network ls. Сетевые драйверы: bridge (по умолчанию), host, overlay, macvlan. Контейнеры в одной bridge-сети видят друг друга по имени."},
    {"id": "doc_08", "title": "Очистка ресурсов", "text": "Со временем накапливаются неиспользуемые образы, контейнеры и тома. Команды очистки: docker system prune (удаляет остановленные контейнеры, неиспользуемые сети и образа без тегов), docker volume prune."},
    {"id": "doc_09", "title": "Переменные окружения", "text": "Для настройки приложения используют переменные окружения. Передача: -e KEY=VALUE или --env-file .env. В Docker Compose указывается в разделе environment. Не храните секреты в открытом виде."},
    {"id": "doc_10", "title": "Multi-stage builds", "text": "Multi-stage build позволяет уменьшить размер финального образа. Используются несколько FROM в одном Dockerfile. В первом этапе собирается приложение, во втором копируются только артефакты. Синтаксис: COPY --from=0."},
    {"id": "doc_11", "title": "Безопасность контейнеров", "text": "Запуск от root не рекомендуется. Используйте USER в Dockerfile, сканируйте образы (trivy, grype), ограничивайте ресурсы (--cpus, --memory), отключайте привилегированный режим (не используйте --privileged без необходимости)."},
    {"id": "doc_12", "title": "Диагностика и логи", "text": "Для отладки: docker logs <container>, docker exec -it <container> sh, docker inspect. Логи пишутся в stdout/stderr. Для продакшена используйте централизованный логинг (Loki, ELK) через драйверы логирования."}
]

print(f"[KB] loaded documents: {len(docs)}")
print("\n[KB] documents examples:")
for d in docs[:3]:
    print(f"- [{d['id']}] {d['title']}: {d['text'][:60]}...")

[KB] loaded documents: 12

[KB] documents examples:
- [doc_01] Что такое Docker: Docker — это платформа для разработки, доставки и запуска пр...
- [doc_02] Dockerfile: Dockerfile — это текстовый файл с инструкциями для сборки об...
- [doc_03] docker build: Команда docker build собирает образ из Dockerfile. Пример: d...


In [4]:
def chunk_documents(docs, chunk_size=25, overlap=8):
    chunks = []
    for doc in docs:
        words = doc["text"].split()
        start = 0
        chunk_idx = 0
        while start < len(words):
            end = start + chunk_size
            chunk_words = words[start:end]
            if not chunk_words:
                break
            chunk_text = " ".join(chunk_words)
            chunks.append({
                "doc_id": doc["id"],
                "chunk_idx": chunk_idx,
                "text": chunk_text
            })
            chunk_idx += 1
            start += chunk_size - overlap
    return chunks

chunks = chunk_documents(docs, chunk_size=25, overlap=8)
print(f"[Chunking] amount of chunks: {len(chunks)}")
print("\n[Chunking] Example for doc_01:")
for c in chunks[:3]:
    print(f"[{c['doc_id']}:{c['chunk_idx']}] {c['text']}")

[Chunking] amount of chunks: 24

[Chunking] Example for doc_01:
[doc_01:0] Docker — это платформа для разработки, доставки и запуска приложений в контейнерах. Контейнеры изолируют приложение и его зависимости от хостовой ОС, обеспечивая воспроизводимость и переносимость.
[doc_01:1] зависимости от хостовой ОС, обеспечивая воспроизводимость и переносимость.
[doc_02:0] Dockerfile — это текстовый файл с инструкциями для сборки образа. Основные команды: FROM, RUN, COPY, CMD, EXPOSE. Образ создаётся послойно, каждый RUN/COPY добавляет новый слой


In [5]:
model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
chunk_texts = [c["text"] for c in chunks]

print("[Embedding] Vector generation")
vectors = model.encode(chunk_texts, normalize_embeddings=True, show_progress_bar=False).astype('float32')
dimension = vectors.shape[1]

print("[FAISS] Index initialization (IndexFlatIP)...")
index = faiss.IndexFlatIP(dimension)
index.add(vectors)
print(f"[FAISS] Size: {index.ntotal}, dim: {dimension}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4501.31it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[Embedding] Vector generation
[FAISS] Index initialization (IndexFlatIP)...
[FAISS] Size: 24, dim: 384


In [8]:
def search_faiss(query, top_k=3):
    q_vec = model.encode([query], normalize_embeddings=True).astype('float32')
    scores, indices = index.search(q_vec, top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1: continue
        chunk = chunks[idx]
        results.append({"doc_id": chunk["doc_id"], "text": chunk["text"], "score": float(score)})
    return results

test_queries = ["как собрать образ из dockerfile", "что такое тома и зачем они нужны", "как запустить контейнер в фоне"]
for q in test_queries:
    res = search_faiss(q, top_k=2)
    print(f"Request: '{q}'")
    for r in res:
        print(f"{r['doc_id']} (score={r['score']:.3f}): {r['text'][:70]}...")

Request: 'как собрать образ из dockerfile'
doc_03 (score=0.906): образа, точка указывает на контекст сборки с Dockerfile....
doc_03 (score=0.834): Команда docker build собирает образ из Dockerfile. Пример: docker buil...
Request: 'что такое тома и зачем они нужны'
doc_01 (score=0.570): зависимости от хостовой ОС, обеспечивая воспроизводимость и переносимо...
doc_11 (score=0.560): отключайте привилегированный режим (не используйте --privileged без не...
Request: 'как запустить контейнер в фоне'
doc_01 (score=0.657): зависимости от хостовой ОС, обеспечивая воспроизводимость и переносимо...
doc_11 (score=0.562): отключайте привилегированный режим (не используйте --privileged без не...


In [9]:
control_queries = [
    {"query": "Как собрать образ из Dockerfile?", "expected": ["doc_02", "doc_03"]},
    {"query": "Запуск контейнера в фоновом режиме", "expected": ["doc_04"]},
    {"query": "Как сохранить данные после остановки контейнера?", "expected": ["doc_06"]},
    {"query": "Настройка переменных окружения", "expected": ["doc_09"]},
    {"query": "Уменьшение размера образа", "expected": ["doc_10"]},
    {"query": "Просмотр логов контейнера", "expected": ["doc_12"]},
    {"query": "Создание изолированной сети для контейнеров", "expected": ["doc_07"]},
    {"query": "Описание мультиконтейнерного приложения", "expected": ["doc_05"]},
    {"query": "Очистка неиспользуемых ресурсов Docker", "expected": ["doc_08"]},
    {"query": "Рекомендации по безопасности запуска", "expected": ["doc_11"]},
]

def evaluate_retrieval(queries, top_k=3, current_chunks=chunks, current_index=index):
    eval_rows = []
    hits = 0
    recalls = []
    
    for item in queries:
        q_vec = model.encode([item["query"]], normalize_embeddings=True).astype('float32')
        _, indices = current_index.search(q_vec, top_k)
        retrieved = [current_chunks[idx]["doc_id"] for idx in indices[0] if idx != -1]
        
        # Убираем дубликаты для корректного расчёта
        retrieved_set = set(retrieved)
        expected_set = set(item["expected"])
        
        hit = 1.0 if len(retrieved_set & expected_set) > 0 else 0.0
        recall = len(retrieved_set & expected_set) / max(len(expected_set), 1)
        
        hits += hit
        recalls.append(recall)
        
        eval_rows.append({
            "query": item["query"],
            "expected_source": "; ".join(item["expected"]),
            "retrieved_sources": "; ".join(retrieved),
            "hit_at_k": hit
        })
    return pd.DataFrame(eval_rows), hits / len(queries), np.mean(recalls)

eval_df, hit_k, recall_k = evaluate_retrieval(control_queries, top_k=3)
print(f"[Eval] hit@3 = {hit_k:.2f}, recall@3 = {recall_k:.2f}")
eval_df.to_csv(os.path.join(ARTIFACTS_DIR, "retrieval_eval.csv"), index=False)
print(f"Сохранено: {ARTIFACTS_DIR}/retrieval_eval.csv")

[Eval] hit@3 = 0.30, recall@3 = 0.30
Сохранено: artifacts/retrieval_eval.csv


In [12]:
# Сравниваем chunk_size=20 и chunk_size=35 при overlap=5
exp_configs = [
    {"name": "small_chunks", "chunk_size": 20, "overlap": 5},
    {"name": "large_chunks", "chunk_size": 35, "overlap": 5}
]

print("[Experiment] Сравнение chunk_size:")
for cfg in exp_configs:
    exp_chunks = chunk_documents(docs, chunk_size=cfg["chunk_size"], overlap=cfg["overlap"])
    exp_vectors = model.encode([c["text"] for c in exp_chunks], normalize_embeddings=True).astype('float32')
    exp_index = faiss.IndexFlatIP(dimension)
    exp_index.add(exp_vectors)
    
    _, h, r = evaluate_retrieval(control_queries, top_k=3, current_chunks=exp_chunks, current_index=exp_index)
    print(f"  {cfg['name']} | chunks: {len(exp_chunks)} | hit@3: {h:.2f} | recall@3: {r:.2f}")

print("\nВывод: Выбран основной параметр chunk_size=25 как баланс между детальностью и контекстной связностью.")

[Experiment] Сравнение chunk_size:
  small_chunks | chunks: 24 | hit@3: 0.40 | recall@3: 0.40
  large_chunks | chunks: 12 | hit@3: 0.50 | recall@3: 0.50

Вывод: Выбран основной параметр chunk_size=25 как баланс между детальностью и контекстной связностью.


In [16]:
# 1. Фиксируем состояние ДО обновления
old_chunks = chunks.copy()
old_index = faiss.IndexFlatIP(dimension)
old_vectors = model.encode([c["text"] for c in old_chunks], normalize_embeddings=True).astype('float32')
old_index.add(old_vectors)

# 2. Добавляем новые документы
new_docs = [
    {"id": "doc_13", "title": "Docker Swarm", "text": "Docker Swarm — нативное решение для оркестрации контейнеров. Позволяет объединять хосты в кластер, управлять службами и автоматически масштабировать реплики. Используется команда docker swarm и docker service."},
    {"id": "doc_14", "title": "Healthcheck", "text": "Healthcheck позволяет Docker проверять работоспособность контейнера. Настраивается через HEALTHCHECK в Dockerfile или в Compose. Если проверка падает, контейнер помечается как unhealthy, но не останавливается автоматически."}
]

# 3. Чанкаем новые документы и обновляем основную базу
new_chunks = chunk_documents(new_docs, chunk_size=25, overlap=8)
new_texts = [c["text"] for c in new_chunks]
new_vectors = model.encode(new_texts, normalize_embeddings=True).astype('float32')

# Инкрементальное обновление основного индекса
index.add(new_vectors)
chunks.extend(new_chunks)

print(f"[Update] Добавлено документов: {len(new_docs)}, новых чанков: {len(new_chunks)}. Всего чанков: {len(chunks)}")

# 4. Запросы для проверки обновления
update_test_queries = [
    {"query": "Оркестрация контейнеров в кластере", "expected": ["doc_13"]},
    {"query": "Как проверить работоспособность контейнера автоматически?", "expected": ["doc_14"]}
]

# 5. Функция сравнения до/после
def compare_before_after(queries, top_k=3):
    rows = []
    for item in queries:
        q_vec = model.encode([item["query"]], normalize_embeddings=True).astype('float32')
        
        # Поиск в старой базе
        _, i_before = old_index.search(q_vec, top_k)
        before = [old_chunks[idx]["doc_id"] for idx in i_before[0] if idx != -1]
        
        # Поиск в обновленной базе
        _, i_after = index.search(q_vec, top_k)
        after = [chunks[idx]["doc_id"] for idx in i_after[0] if idx != -1]
        
        changed = before != after
        rows.append({
            "query": item["query"],
            "before_retrieved_sources": "; ".join(before) if before else "none",
            "after_retrieved_sources": "; ".join(after) if after else "none",
            "changed": changed
        })
    return pd.DataFrame(rows)

# 6. Выполняем сравнение и сохраняем артефакт
update_df = compare_before_after(update_test_queries, top_k=3)
update_df.to_csv(os.path.join(ARTIFACTS_DIR, "retrieval_before_after_update.csv"), index=False)
print("Сохранено: retrieval_before_after_update.csv")
print("\nСравнение retrieval до и после обновления:")
print(update_df.to_string(index=False))

[Update] Добавлено документов: 2, новых чанков: 4. Всего чанков: 36
Сохранено: retrieval_before_after_update.csv

Сравнение retrieval до и после обновления:
                                                    query before_retrieved_sources after_retrieved_sources  changed
                       Оркестрация контейнеров в кластере   doc_01; doc_09; doc_14  doc_01; doc_09; doc_14    False
Как проверить работоспособность контейнера автоматически?   doc_14; doc_14; doc_01  doc_14; doc_14; doc_14     True


In [14]:
def mini_rag(query, top_k=3, chunks=chunks, index=index, model=model):
    # 1. Retrieval
    q_vec = model.encode([query], normalize_embeddings=True).astype('float32')
    _, indices = index.search(q_vec, top_k)
    retrieved = [chunks[idx] for idx in indices[0] if idx != -1]
    
    # 2. Context assembly
    context_parts = [f"[{r['doc_id']}] {r['text']}" for r in retrieved]
    context = "\n\n".join(context_parts)
    
    # 3. Generation 
    answer = f"На основе предоставленной документации: {context_parts[0] if context_parts else 'Нет релевантных данных'}"
    if len(context_parts) > 1:
        answer += f" Дополнительно: {context_parts[1]}..."
        
    sources = [r["doc_id"] for r in retrieved]
    return {"question": query, "answer": answer, "retrieved_sources": "; ".join(sources)}

rag_questions = [
    "Как пробросить порт из контейнера на хост?",
    "Зачем нужен multi-stage build?",
    "Как посмотреть логи запущенного контейнера?",
    "Что делать, если контейнер помечен как unhealthy?"
]

rag_results = [mini_rag(q, top_k=3) for q in rag_questions]
rag_df = pd.DataFrame(rag_results)
rag_df.to_csv(os.path.join(ARTIFACTS_DIR, "rag_examples.csv"), index=False)
print("Сохранено: rag_examples.csv")
print("\nMini-RAG примеры:")
for r in rag_results:
    print(f"{r['question']}\n{r['answer'][:120]}...\nИсточники: {r['retrieved_sources']}\n---")

Сохранено: rag_examples.csv

Mini-RAG примеры:
Как пробросить порт из контейнера на хост?
На основе предоставленной документации: [doc_01] зависимости от хостовой ОС, обеспечивая воспроизводимость и переносимос...
Источники: doc_01; doc_14; doc_02
---
Зачем нужен multi-stage build?
На основе предоставленной документации: [doc_10] Multi-stage build позволяет уменьшить размер финального образа. Использ...
Источники: doc_10; doc_08; doc_03
---
Как посмотреть логи запущенного контейнера?
На основе предоставленной документации: [doc_01] зависимости от хостовой ОС, обеспечивая воспроизводимость и переносимос...
Источники: doc_01; doc_14; doc_02
---
Что делать, если контейнер помечен как unhealthy?
На основе предоставленной документации: [doc_14] контейнер помечается как unhealthy, но не останавливается автоматически...
Источники: doc_14; doc_14; doc_09
---


### Анализ ошибок mini-RAG
| Вопрос | Проблема | Причина |
|--------|----------|---------|
| `Как пробросить порт из контейнера на хост?` | Ответ обрезан на ключевой детали | Вектор поиска нашёл чанк с `-p`, но без полного синтаксиса `host:container` |
| `Как настроить оркестрацию в Kubernetes?` | RAG ответил на основе Swarm | В базе нет K8s, модель экстраполировала из `doc_13` (Swarm) |
| `Где хранятся логи по умолчанию?` | Контекст не содержит точного пути | В `doc_12` указано `stdout/stderr`, но не путь на диске хоста |